### Create anndata object from re-mapped E-MTAB-9543 samples
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Creation date:** 4th of November 2024
- **Last modified date:** 4th of November 2024

### Import packages

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sci
import anndata as ad
from scipy import io,sparse
import os
import muon as mu
from datetime import datetime

+ We want to save the spliced/unspliced counts in our anndata object

In [2]:
def starsolo_velocity_anndata(input_dir):
    """
    input directory should contain barcodes.tsv, features.tsv with 3 mtx from spliced, ambigious, unspliced
    """
    obs = pd.read_csv(os.path.join(input_dir,'barcodes.tsv'), header = None, index_col = 0)
    # Remove index column name to make it compliant with the anndata format
    obs.index.name = None

    var = pd.read_csv(os.path.join(input_dir,"features.tsv"), sep='\t',names = ('gene_ids', 'feature_types'), index_col = 1)
    var.index.name = None

    spliced=sci.sparse.csr_matrix(sci.io.mmread(os.path.join(input_dir,"spliced.mtx")).T)
    ambiguous=sci.sparse.csr_matrix(sci.io.mmread(os.path.join(input_dir,"ambiguous.mtx")).T)
    unspliced=sci.sparse.csr_matrix(sci.io.mmread(os.path.join(input_dir,"unspliced.mtx")).T)
    adata=ad.AnnData(X=spliced,obs=obs,var=var,layers={'spliced':spliced,"ambiguous":ambiguous,"unspliced":unspliced})
    adata.var_names_make_unique()
    return adata

+ Upload sample description dataframe

In [3]:
samples = pd.read_csv("raw_fastq_files/Elmentaite_2021/metadata/E-MTAB-9543.sdrf.txt", sep = "\t")

+ Base path for remapped samples

In [4]:
base_path = 'raw_data/Elmentaite_2021/remapped_adult_data_EMTAB-9543_starsolo'

In [5]:
ann_data_list = []
failed_samples = []

for sample_name in samples['Extract Name']:
    try:
        # Try loading the AnnData object from the UMI10_output path
        sample_path = f"{base_path}/{sample_name}/UMI10_output/Velocyto/raw"
        sample_name_adata = starsolo_velocity_anndata(sample_path)

        # Create the cell_id column
        sample_name_adata.obs['Extract Name'] = sample_name

        ann_data_list.append(sample_name_adata)
    except FileNotFoundError:
        try:
            # If not found in UMI10_output, try loading from the UMI12_output path
            sample_path = f"{base_path}/{sample_name}/UMI12_output/Velocyto/raw"
            sample_name_adata = starsolo_velocity_anndata(sample_path)

            # Create the cell_id column
            sample_name_adata.obs['Extract Name'] = sample_name

            ann_data_list.append(sample_name_adata)
        except FileNotFoundError:
            # If sample is not found in either path, add it to the failed_samples list
            failed_samples.append(sample_name)
            print(f"Sample {sample_name} not found in both UMI10 and UMI12 paths, skipping.")

# Merge all AnnData objects into one, if there are any
if ann_data_list:
    combined_adata = ann_data_list[0].concatenate(ann_data_list[1:], join='outer')
else:
    combined_adata = None
    print("No valid AnnData objects found to merge.")

# List samples that were not processed
if failed_samples:
    print("The following samples were not processed:")
    for sample in failed_samples:
        print(sample)


/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/utils.py:260: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-4', 'SNORD116-5']
  warnings.warn(
/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are

Sample Human_colon_16S8000484 not found in both UMI10 and UMI12 paths, skipping.
Sample Human_colon_16S8000484 not found in both UMI10 and UMI12 paths, skipping.
Sample Human_colon_16S8000484 not found in both UMI10 and UMI12 paths, skipping.


/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/utils.py:260: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-4', 'SNORD116-5']
  warnings.warn(
/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are

The following samples were not processed:
Human_colon_16S8000484
Human_colon_16S8000484
Human_colon_16S8000484


In [6]:
combined_adata.obs['barcode'] = combined_adata.obs.index.copy()

In [7]:
combined_adata.obs

,Extract Name,batch,barcode
AAACCTGAGAAACCAT-0,Human_colon_16S8000471,0,AAACCTGAGAAACCAT-0
AAACCTGAGAAACCGC-0,Human_colon_16S8000471,0,AAACCTGAGAAACCGC-0
AAACCTGAGAAACCTA-0,Human_colon_16S8000471,0,AAACCTGAGAAACCTA-0
AAACCTGAGAAACGAG-0,Human_colon_16S8000471,0,AAACCTGAGAAACGAG-0
AAACCTGAGAAACGCC-0,Human_colon_16S8000471,0,AAACCTGAGAAACGCC-0
...,...,...,...
TTTGTCATCTTTACAC-188,Human_colon_16S8159194,188,TTTGTCATCTTTACAC-188
TTTGTCATCTTTACGT-188,Human_colon_16S8159194,188,TTTGTCATCTTTACGT-188
TTTGTCATCTTTAGGG-188,Human_colon_16S8159194,188,TTTGTCATCTTTAGGG-188
TTTGTCATCTTTAGTC-188,Human_colon_16S8159194,188,TTTGTCATCTTTAGTC-188


In [8]:
# remove dublicates in samples['Extract Name']
samples = samples.drop_duplicates(subset=['Extract Name'])

,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[age],Unit[time unit],Term Source REF,Term Accession Number,Characteristics[developmental stage],Characteristics[sex],...,Comment[ENA_EXPERIMENT],Scan Name,Comment[SUBMITTED_FILE_NAME],Comment[ENA_RUN],Comment[FASTQ_URI],Comment[read_type],Comment[read_index],Factor Value[immunophenotype],Factor Value[organism part],Factor Value[single cell identifier]
0,Human_colon_16S8000471,ERS12382553,SAMEA110284943,Homo sapiens,65 to 70,year,EFO,UO_0000036,adult,male,...,ERX9467918,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,ERR9924225,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,caecum,A34-CAE-1-SC-45N-1
3,Human_colon_16S8000473,ERS12382554,SAMEA110284944,Homo sapiens,65 to 70,year,EFO,UO_0000036,adult,male,...,ERX9467919,Human_colon_16S8000473_S1_L001_I1_001.fastq.gz,Human_colon_16S8000473_S1_L001_I1_001.fastq.gz,ERR9924226,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,ascending colon,A34-ACL-1-SC-45N-1
6,Human_colon_16S8000475,ERS12382555,SAMEA110284945,Homo sapiens,65 to 70,year,EFO,UO_0000036,adult,male,...,ERX9467920,Human_colon_16S8000475_S1_L001_I1_001.fastq.gz,Human_colon_16S8000475_S1_L001_I1_001.fastq.gz,ERR9924227,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,transverse colon,A34-TCL-1-SC-45N-1
9,Human_colon_16S8000477,ERS12382556,SAMEA110284946,Homo sapiens,65 to 70,year,EFO,UO_0000036,adult,male,...,ERX9467921,Human_colon_16S8000477_S1_L001_I1_001.fastq.gz,Human_colon_16S8000477_S1_L001_I1_001.fastq.gz,ERR9924228,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,descending colon,A34-DCL-1-SC-45N-1
12,Human_colon_16S8000479,ERS12382557,SAMEA110284947,Homo sapiens,65 to 70,year,EFO,UO_0000036,adult,male,...,ERX9467922,Human_colon_16S8000479_S1_L001_I1_001.fastq.gz,Human_colon_16S8000479_S1_L001_I1_001.fastq.gz,ERR9924229,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,sigmoid colon,A34-SCL-1-SC-45N-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,Human_colon_16S8123921,ERS12382612,SAMEA110285002,Homo sapiens,45 to 50,year,EFO,UO_0000036,adult,male,...,ERX9467977,Human_colon_16S8123921_S1_L001_I1_001.fastq.gz,Human_colon_16S8123921_S1_L001_I1_001.fastq.gz,ERR9924284,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,unsorted,mesenteric lymph node,A39-MLN--SC-1
180,Human_colon_16S8159191,ERS12382613,SAMEA110285003,Homo sapiens,25 to 30,year,EFO,UO_0000036,adult,male,...,ERX9467978,Human_colon_16S8159191_S1_L001_I1_001.fastq.gz,Human_colon_16S8159191_S1_L001_I1_001.fastq.gz,ERR9924285,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 positive,transverse colon,A32-TCL-0-SC-45P-1
183,Human_colon_16S8159192,ERS12382614,SAMEA110285004,Homo sapiens,25 to 30,year,EFO,UO_0000036,adult,male,...,ERX9467979,Human_colon_16S8159192_S1_L001_I1_001.fastq.gz,Human_colon_16S8159192_S1_L001_I1_001.fastq.gz,ERR9924286,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 positive,appendix,A32-APD-0-SC-45P-1
186,Human_colon_16S8159193,ERS12382615,SAMEA110285005,Homo sapiens,25 to 30,year,EFO,UO_0000036,adult,male,...,ERX9467980,Human_colon_16S8159193_S1_L001_I1_001.fastq.gz,Human_colon_16S8159193_S1_L001_I1_001.fastq.gz,ERR9924287,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 positive,caecum,A32-CAE-0-SC-45P-1


In [10]:
combined_adata.obs = combined_adata.obs.merge(samples, on='Extract Name', how='left', suffixes=('', '_y'))

combined_adata.obs = combined_adata.obs.loc[:, ~combined_adata.obs.columns.str.endswith('_y')]

In [11]:
combined_adata.obs.index = combined_adata.obs['barcode']

In [12]:
combined_adata.obs

,Extract Name,batch,barcode,Source Name,Comment[ENA_SAMPLE],Comment[BioSD_SAMPLE],Characteristics[organism],Characteristics[age],Unit[time unit],Term Source REF,...,Comment[ENA_EXPERIMENT],Scan Name,Comment[SUBMITTED_FILE_NAME],Comment[ENA_RUN],Comment[FASTQ_URI],Comment[read_type],Comment[read_index],Factor Value[immunophenotype],Factor Value[organism part],Factor Value[single cell identifier]
barcode,,,,,,,,,,,,,,,,,,,,,
AAACCTGAGAAACCAT-0,Human_colon_16S8000471,0,AAACCTGAGAAACCAT-0,Human_colon_16S8000471,ERS12382553,SAMEA110284943,Homo sapiens,65 to 70,year,EFO,...,ERX9467918,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,ERR9924225,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,caecum,A34-CAE-1-SC-45N-1
AAACCTGAGAAACCGC-0,Human_colon_16S8000471,0,AAACCTGAGAAACCGC-0,Human_colon_16S8000471,ERS12382553,SAMEA110284943,Homo sapiens,65 to 70,year,EFO,...,ERX9467918,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,ERR9924225,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,caecum,A34-CAE-1-SC-45N-1
AAACCTGAGAAACCTA-0,Human_colon_16S8000471,0,AAACCTGAGAAACCTA-0,Human_colon_16S8000471,ERS12382553,SAMEA110284943,Homo sapiens,65 to 70,year,EFO,...,ERX9467918,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,ERR9924225,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,caecum,A34-CAE-1-SC-45N-1
AAACCTGAGAAACGAG-0,Human_colon_16S8000471,0,AAACCTGAGAAACGAG-0,Human_colon_16S8000471,ERS12382553,SAMEA110284943,Homo sapiens,65 to 70,year,EFO,...,ERX9467918,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,ERR9924225,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,caecum,A34-CAE-1-SC-45N-1
AAACCTGAGAAACGCC-0,Human_colon_16S8000471,0,AAACCTGAGAAACGCC-0,Human_colon_16S8000471,ERS12382553,SAMEA110284943,Homo sapiens,65 to 70,year,EFO,...,ERX9467918,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,Human_colon_16S8000471_S1_L001_I1_001.fastq.gz,ERR9924225,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,CD45 negative,caecum,A34-CAE-1-SC-45N-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCATCTTTACAC-188,Human_colon_16S8159194,188,TTTGTCATCTTTACAC-188,Human_colon_16S8159194,ERS12382616,SAMEA110285006,Homo sapiens,25 to 30,year,EFO,...,ERX9467981,Human_colon_16S8159194_S1_L001_I1_001.fastq.gz,Human_colon_16S8159194_S1_L001_I1_001.fastq.gz,ERR9924288,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,unsorted,mesenteric lymph node,A32-MLN-0-SC-1
TTTGTCATCTTTACGT-188,Human_colon_16S8159194,188,TTTGTCATCTTTACGT-188,Human_colon_16S8159194,ERS12382616,SAMEA110285006,Homo sapiens,25 to 30,year,EFO,...,ERX9467981,Human_colon_16S8159194_S1_L001_I1_001.fastq.gz,Human_colon_16S8159194_S1_L001_I1_001.fastq.gz,ERR9924288,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,unsorted,mesenteric lymph node,A32-MLN-0-SC-1
TTTGTCATCTTTAGGG-188,Human_colon_16S8159194,188,TTTGTCATCTTTAGGG-188,Human_colon_16S8159194,ERS12382616,SAMEA110285006,Homo sapiens,25 to 30,year,EFO,...,ERX9467981,Human_colon_16S8159194_S1_L001_I1_001.fastq.gz,Human_colon_16S8159194_S1_L001_I1_001.fastq.gz,ERR9924288,ftp://ftp.sra.ebi.ac.uk/vol1/run/ERR992/ERR992...,sample_barcode,index1,unsorted,mesenteric lymph node,A32-MLN-0-SC-1


In [13]:
project = 'gut'
species = 'hs'
atribute = 'Elementaite2021_E-MTAB-9543'
name = 'AM'
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')
counts = 'raw'

combined_adata.uns['processing_history'] = []
combined_adata.uns['processing_history']={
    'timestamp': timestamp,
    'step': 'create raw anndata after mapping, no filtering'}


combined_adata.write_h5ad(f"raw_data/Elmentaite_2021/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5ad")

In [15]:
mdata = mu.MuData({'rna': combined_adata})
mdata.write(f"raw_data/Elmentaite_2021/{project}_{species}_{atribute}_{name}_{timestamp}_{counts}.h5mu")

/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/mudata/_core/mudata.py:1531: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/amaguza/.local/share/hatch/env/virtual/single-cell-project/HC5eoTg7/single_cell_project/lib/python3.10/site-packages/mudata/_core/mudata.py:1429: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=joi